In [32]:
from sklearn.preprocessing import robust_scale, LabelEncoder
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd 
from pathlib import Path

## 10% Sample Creation 

In [33]:
PROCESSED_DIR = Path("../data/processed")
OUTPUT_DIR = Path("../data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# List all cleaned batch files
batch_files = sorted(PROCESSED_DIR.glob("cleaned_batch*.parquet"))
print(f"Found {len(batch_files)} batch files")

sample_dfs = []

for file in batch_files:
    print(f"\nProcessing {file.name}...")
    df_full = pd.read_parquet(file)
    print(f"  Total rows: {len(df_full):,}")
    
    # Keep ONLY complete rows (passenger_count not null)
    df_complete = df_full[df_full['passenger_count'].notna()].copy()
    print(f"  Complete rows: {len(df_complete):,} ({len(df_complete)/len(df_full)*100:.1f}%)")
    
    # Take 10% from complete rows only
    df_sample = df_complete.sample(frac=0.1, random_state=42)
    sample_dfs.append(df_sample)
    print(f"  Sampled: {len(df_sample):,} rows")

# Combine all samples
df_model = pd.concat(sample_dfs, ignore_index=True)
print(f"\n Final model dataset: {len(df_model):,} rows")

# Save to parquet
output_path = OUTPUT_DIR / "model_sample_10pct_complete.parquet"
df_model.to_parquet(output_path, index=False)
print(f"Saved to {output_path}")



Found 4 batch files

Processing cleaned_batch1_jan_mar.parquet...
  Total rows: 10,385,509
  Complete rows: 8,617,867 (83.0%)
  Sampled: 861,787 rows

Processing cleaned_batch2_apr_jun.parquet...
  Total rows: 11,633,505
  Complete rows: 9,228,225 (79.3%)
  Sampled: 922,822 rows

Processing cleaned_batch3_jul_sep.parquet...
  Total rows: 10,513,539
  Complete rows: 8,232,659 (78.3%)
  Sampled: 823,266 rows

Processing cleaned_batch4_oct_dec.parquet...
  Total rows: 11,636,893
  Complete rows: 9,245,719 (79.5%)
  Sampled: 924,572 rows

 Final model dataset: 3,532,447 rows
Saved to ..\data\processed\model_sample_10pct_complete.parquet


In [34]:
df = pd.read_parquet(PROCESSED_DIR / "model_sample_10pct_complete.parquet")
df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,2,2025-03-08 21:07:12,2025-03-08 21:10:39,4.0,0.25,1.0,N,164,164,4,5.1,1.00,0.5,0.00,0.00,1.0,10.85,2.5,0.00,0.75
1,1,2025-02-06 08:38:46,2025-02-06 08:47:43,1.0,1.90,1.0,N,234,231,1,11.4,3.25,0.5,1.94,0.00,1.0,18.09,2.5,0.00,0.75
2,2,2025-03-16 17:11:50,2025-03-16 17:16:31,1.0,0.78,1.0,N,113,234,1,6.5,0.00,0.5,1.50,0.00,1.0,12.75,2.5,0.00,0.75
3,2,2025-02-12 01:02:48,2025-02-12 01:33:39,1.0,18.17,2.0,N,132,186,1,70.0,0.00,0.5,16.34,6.94,1.0,99.78,2.5,1.75,0.75
4,1,2025-01-31 11:40:47,2025-01-31 11:51:04,1.0,0.80,1.0,N,234,170,1,9.3,3.25,0.5,3.50,0.00,1.0,17.55,2.5,0.00,0.75


### Create Time-Based Features

In [35]:
df['pickup_hour'] = df['tpep_pickup_datetime'].dt.hour
df['pickup_minute'] = df['tpep_pickup_datetime'].dt.minute
df['pickup_dayofweek'] = df['tpep_pickup_datetime'].dt.dayofweek  # 0=Monday, 6=Sunday
df['pickup_day'] = df['tpep_pickup_datetime'].dt.day
df['pickup_month'] = df['tpep_pickup_datetime'].dt.month
df['is_weekend'] = df['pickup_dayofweek'].isin([5, 6]).astype(int)  # Saturday=5, Sunday=6
# Rush hour: 7-9 AM and 5-7 PM on weekdays
df['is_rush_hour'] = (
    ((df['pickup_hour'].between(7, 9)) | (df['pickup_hour'].between(17, 19))) & 
    (df['is_weekend'] == 0)
).astype(int)
# Night: 10 PM - 5 AM
df['is_night'] = df['pickup_hour'].between(22, 23) | df['pickup_hour'].between(0, 5).astype(int)

In [36]:
# Trip duration and speed
df['trip_duration_min'] = (df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']).dt.total_seconds() / 60
df['trip_duration_hours'] = df['trip_duration_min'] / 60

# Speed miles per hour
df['trip_speed_mph'] = df['trip_distance'] / df['trip_duration_hours'].replace(0, np.nan)

# Handle infinite values
df['trip_speed_mph'] = df['trip_speed_mph'].replace([np.inf, -np.inf], np.nan)
df['trip_speed_mph'] = df['trip_speed_mph'].fillna(df['trip_speed_mph'].median())


### Payment & Categorical Encodings

In [37]:
# One-hot encode 
payment_dummies = pd.get_dummies(df['payment_type'], prefix='payment')
df = pd.concat([df, payment_dummies], axis=1)

In [38]:
ratecode_dummies = pd.get_dummies(df['RatecodeID'], prefix='ratecode')
df = pd.concat([df, ratecode_dummies], axis=1)

In [39]:
df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,payment_2,payment_3,payment_4,ratecode_1.0,ratecode_2.0,ratecode_3.0,ratecode_4.0,ratecode_5.0,ratecode_6.0,ratecode_99.0
0,2,2025-03-08 21:07:12,2025-03-08 21:10:39,4.0,0.25,1.0,N,164,164,4,...,False,False,True,True,False,False,False,False,False,False
1,1,2025-02-06 08:38:46,2025-02-06 08:47:43,1.0,1.90,1.0,N,234,231,1,...,False,False,False,True,False,False,False,False,False,False
2,2,2025-03-16 17:11:50,2025-03-16 17:16:31,1.0,0.78,1.0,N,113,234,1,...,False,False,False,True,False,False,False,False,False,False
3,2,2025-02-12 01:02:48,2025-02-12 01:33:39,1.0,18.17,2.0,N,132,186,1,...,False,False,False,False,True,False,False,False,False,False
4,1,2025-01-31 11:40:47,2025-01-31 11:51:04,1.0,0.80,1.0,N,234,170,1,...,False,False,False,True,False,False,False,False,False,False


### Creating Aggregated Dataset for Demand Forecasting

In [40]:
df = df.sort_values('tpep_pickup_datetime')

# Create hour-floor column (lowercase 'h')
df['pickup_hour_floor'] = df['tpep_pickup_datetime'].dt.floor('h')

# Aggregate by hour and pickup zone
demand = df.groupby(['pickup_hour_floor', 'PULocationID']).agg(
    trip_count=('fare_amount', 'count'),
    avg_fare=('fare_amount', 'mean'),
    avg_distance=('trip_distance', 'mean'),
    avg_duration_min=('trip_duration_min', 'mean')
).reset_index()

# Time features
demand['hour'] = demand['pickup_hour_floor'].dt.hour
demand['dayofweek'] = demand['pickup_hour_floor'].dt.dayofweek
demand['month'] = demand['pickup_hour_floor'].dt.month
demand['is_weekend'] = demand['dayofweek'].isin([5,6]).astype(int)

# Sort for lags
demand = demand.sort_values(['PULocationID', 'pickup_hour_floor'])

# Lag features
demand['trips_last_hour'] = demand.groupby('PULocationID')['trip_count'].shift(1)
demand['trips_same_hour_yesterday'] = demand.groupby('PULocationID')['trip_count'].shift(24)
demand['trips_same_hour_last_week'] = demand.groupby('PULocationID')['trip_count'].shift(168)
demand['trips_rolling_7h'] = demand.groupby('PULocationID')['trip_count'].transform(lambda x: x.rolling(7, min_periods=1).mean())

# Drop NaN 
demand = demand.dropna(subset=['trips_last_hour', 'trips_same_hour_yesterday', 'trips_same_hour_last_week'])


In [41]:
demand.head()

,pickup_hour_floor,PULocationID,trip_count,avg_fare,avg_distance,avg_duration_min,hour,dayofweek,month,is_weekend,trips_last_hour,trips_same_hour_yesterday,trips_same_hour_last_week,trips_rolling_7h
454059,2025-12-12 06:00:00,3,1,54.5,14.2,84.066667,6,4,12,0,1.0,1.0,1.0,1.0
454438,2025-12-12 11:00:00,3,1,37.5,9.7,72.183333,11,4,12,0,1.0,1.0,1.0,1.0
459781,2025-12-16 07:00:00,3,1,41.5,13.0,51.050000,7,1,12,0,1.0,1.0,1.0,1.0
460043,2025-12-16 10:00:00,3,1,36.5,9.4,57.016667,10,1,12,0,1.0,1.0,1.0,1.0
461203,2025-12-17 06:00:00,3,1,51.5,14.9,48.666667,6,2,12,0,1.0,1.0,1.0,1.0


### Dropping unuseful columns

In [42]:
drop_these = [
    'tpep_dropoff_datetime', 'pickup_minute', 'pickup_day', 
    'pickup_hour_floor', 'ratecode_99.0', 'store_and_fwd_flag', 'VendorID'
]

df = df.drop(columns=drop_these, errors='ignore')

In [43]:
print(df.columns)
print(demand.columns)

Index(['tpep_pickup_datetime', 'passenger_count', 'trip_distance',
       'RatecodeID', 'PULocationID', 'DOLocationID', 'payment_type',
       'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount',
       'improvement_surcharge', 'total_amount', 'congestion_surcharge',
       'Airport_fee', 'cbd_congestion_fee', 'pickup_hour', 'pickup_dayofweek',
       'pickup_month', 'is_weekend', 'is_rush_hour', 'is_night',
       'trip_duration_min', 'trip_duration_hours', 'trip_speed_mph',
       'payment_1', 'payment_2', 'payment_3', 'payment_4', 'ratecode_1.0',
       'ratecode_2.0', 'ratecode_3.0', 'ratecode_4.0', 'ratecode_5.0',
       'ratecode_6.0'],
      dtype='str')
Index(['pickup_hour_floor', 'PULocationID', 'trip_count', 'avg_fare',
       'avg_distance', 'avg_duration_min', 'hour', 'dayofweek', 'month',
       'is_weekend', 'trips_last_hour', 'trips_same_hour_yesterday',
       'trips_same_hour_last_week', 'trips_rolling_7h'],
      dtype='str')


### Saving data

In [44]:
df.to_parquet("../data/processed/prediction_anomaly_data.parquet", index=False)
demand.to_parquet("../data/processed/forecasting_data.parquet", index=False)